# HateMirage @ ICON 2026 - 2xT4 QLoRA (dual-GPU, HF-sync, resume, auto-eval)

**Tasks:** A=Target span - B=Intent+Implication generation - **Metrics:** SBERT-cosine + ROUGE-L F1 (greedy T=0)  
**Setup:** Kaggle accelerator `GPU T4 x2` - Internet ON - Secrets: `HF_TOKEN` (write) - Optional: `WANDB_API_KEY`  
**Flow:** env-check -> install -> HF login+config -> data -> RAG -> torchrun-DDP train (both T4s) -> Hub push -> inference -> eval -> Codabench zips.  
If the session hits the 9h limit or converges early, training saves + pushes and the notebook jumps straight to eval - never lose a run. Resume anytime from `HF_REPO`.

Paper: https://arxiv.org/abs/2603.02684 - Task: https://sai-kartheek-reddy.github.io/HateMirage-ICON2026/ - Codabench: https://www.codabench.org/competitions/17783/

In [ ]:
# CELL 1 - env check: must see 2x Tesla T4
import torch, os, time, shutil
T0 = time.time()
print('torch', torch.__version__, '| cuda?', torch.cuda.is_available())
NGPU = torch.cuda.device_count()
print('GPUs:', NGPU)
for i in range(NGPU):
    print(i, torch.cuda.get_device_name(i), f"{torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB")
!nvidia-smi -L
!df -h /kaggle/working | tail -1
assert NGPU >= 1, 'Enable GPU T4 x2 under Settings > Accelerator'
DUAL = NGPU >= 2
print('DUAL-GPU mode:', DUAL)
OUT = '/kaggle/working/hatemirage_out'
os.makedirs(OUT, exist_ok=True)
os.environ["OUT"] = OUT
print('OUT:', OUT)

In [ ]:
# CELL 2 - deps (Kaggle default torch 2.x; keep bnb compatible)
!pip install -q -U "transformers>=4.44" accelerate peft trl bitsandbytes "datasets>=2.20" sentence-transformers rouge-score faiss-cpu huggingface_hub pandas scikit-learn openpyxl 2>&1 | tail -2
import transformers, peft, trl, bitsandbytes
print('transformers', transformers.__version__)

In [ ]:
# CELL 3 - secrets (resilient: smoke mode if HF_TOKEN absent) + run config
import os
HF_TOKEN_HARDCODED = "hf_YOUR_TOKEN_HERE"  # private fallback; Secrets win
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN secret: present')
except Exception as e:
    HF_TOKEN = HF_TOKEN_HARDCODED
    print('HF_TOKEN secret: ABSENT -> using built-in fallback.', str(e)[:100])
    print('HF_TOKEN: unexpected empty (cleared fallback).', str(e)[:100])
os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '0'
HAS_HF = bool(HF_TOKEN)
if HAS_HF:
    get_ipython().system('huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential 2>&1 | tail -1')

# ---- EDIT: your Hub namespace + model choice ----
HF_USER = 'blackpanter_252'
if HAS_HF:
    try:
        from huggingface_hub import whoami as _who3
        HF_USER = _who3(token=HF_TOKEN).get('name', HF_USER)
        print('HF user:', HF_USER)
    except Exception as _e3:
        print('whoami failed, keep HF_USER:', HF_USER, str(_e3)[:100])
MODEL_ID = 'microsoft/Phi-3-mini-128k-instruct'
SMOKE_MODEL = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
EFFECTIVE_MODEL = MODEL_ID if HAS_HF else SMOKE_MODEL
HF_REPO  = f'{HF_USER}/hatemirage-phi3-qlora'
WITH_RAG, MAX_RUNTIME_H = True, 8.2
EPOCHS, BS, GRAD_ACCUM, LORA_R = (3, 2, 8, 64) if HAS_HF else (1, 2, 2, 8)
print('model:', EFFECTIVE_MODEL, '| hub:', HF_REPO if HAS_HF else '(skipped, no token)', '| RAG:', WITH_RAG)
if HAS_HF:
    from huggingface_hub import whoami
    print(whoami(token=HF_TOKEN)['name'])


In [ ]:
# CELL 4 - official task data (Development + Evaluation phases, pinned SHAs)
# Train.xlsx 04cb07ab | Val-for-dev a81fe96d | Val-Data-with-labels 838c3f83 | test-data-for-eval d76f741f
import glob, pandas as pd, os, urllib.request
GH = 'https://raw.githubusercontent.com/Sai-Kartheek-Reddy/HateMirage-ICON2026/main/'
OFFICIAL = {
 'train': 'Development%20Phase/Train.xlsx',
 'devval': 'Development%20Phase/Val-for-dev.xlsx',
 'vallab': 'Evaluation%20Phase/Val-Data-with-labels.xlsx',
 'test': 'Evaluation%20Phase/test-data-for-eval.xlsx',
}
os.makedirs(f'{OUT}/official', exist_ok=True)
def fetch_official(key):
    p = f'{OUT}/official/{OFFICIAL[key].split("/")[-1]}'
    if not os.path.exists(p):
        urllib.request.urlretrieve(GH + OFFICIAL[key], p)
        print('downloaded', p)
    return p
def load_off(p, labeled):
    df = pd.read_excel(p, engine='openpyxl')
    df.columns = [c.strip() for c in df.columns]
    out = pd.DataFrame({'Index': df['Index'], 'comment': df['Comments'].astype(str)})
    if labeled:
        out['target'] = df['Target'].astype(str)
        out['intent'] = df['Intent'].astype(str)
        out['implication'] = df['Implication'].astype(str)
    return out.dropna().reset_index(drop=True)
try:
    train_df = load_off(fetch_official('train'), True)
    val_df = load_off(fetch_official('vallab'), True)
    devval_df = load_off(fetch_official('devval'), False)
    test_df = load_off(fetch_official('test'), False)
    print('OFFICIAL data:', len(train_df), len(val_df), len(devval_df), len(test_df))
except Exception as e:
    print('official fetch failed, fallback chain:', str(e)[:200])
    cands = glob.glob('/kaggle/input/**/*.*', recursive=True)[:10]
    print('kaggle inputs:', cands)
    smoke = {'Index': range(12), 'comment': ['The virus came from [Country] labs; they engineered it to destroy us.']*12,
             'target': ['[Country]']*12, 'intent': ['To accuse [Country] of building a bioweapon.']*12,
             'implication': ['Could incite hostility against [Country] citizens.']*12}
    train_df = pd.DataFrame(smoke); val_df = train_df.iloc[:4].copy()
    devval_df = train_df[['Index','comment']].iloc[:4].copy()
    test_df = train_df[['Index','comment']].iloc[:4].copy()
for n, df in [('train',train_df),('val',val_df),('devval',devval_df),('test',test_df)]:
    print(n, df.shape, list(df.columns))
    display(df.head(2))
train_df.to_csv(f'{OUT}/train_norm.csv', index=False); val_df.to_csv(f'{OUT}/val_norm.csv', index=False)
devval_df.to_csv(f'{OUT}/devval_norm.csv', index=False); test_df.to_csv(f'{OUT}/test_norm.csv', index=False)
print('normalized ->', OUT)
